# 🏦 ECB Monetary Policy Surprises — Multi-Agent LLM Pipeline

**Pipeline:** Agent IM → Agent IB → Agent II → Agent III  
**Period:** 1999–2025  
**LLM:** Gemini 2.5 Flash  

---
## Cell 1 — Setup & Installation

In [ ]:
df_draghi = df[df['meeting_date'].between('2011-11-01', '2019-10-31')]
print(f"Max Draghi: {df_draghi['dfr_surprise_mech'].max():.3f}bp")
print(df_draghi.nlargest(3, 'dfr_surprise_mech')[['meeting_date','dfr_surprise_mech']])

NameError: name 'df' is not defined

In [ ]:
# ============================================================
# CELL 1 — SETUP, INSTALLATION, CONFIGURATION
# ============================================================

!pip install pdfplumber beautifulsoup4 google-genai pydantic tqdm requests -q

import os, sys, json, re, time, logging
from pathlib import Path
from datetime import date, datetime, timedelta
from typing import Optional, List, Dict
from dataclasses import dataclass

import pandas as pd
import numpy as np
import pdfplumber
from bs4 import BeautifulSoup
from tqdm import tqdm
from google import genai
from pydantic import BaseModel, Field, model_validator

# ── Google Drive ──────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# ── Logging ───────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
    datefmt='%H:%M:%S'
)
log = logging.getLogger('ecb_pipeline')

# ── Config ────────────────────────────────────────────────────
class Config:
    DRIVE_ROOT          = Path('/content/drive/MyDrive/ECB_Research/ECB_Project')
    ACCOUNTS_DIR        = DRIVE_ROOT / 'data/raw/accounts'
    ECON_BULLETINS_DIR  = DRIVE_ROOT / 'data/raw/economic_bulletins'
    MONTHLY_BULL_DIR    = DRIVE_ROOT / 'data/raw/monthly_bulletins'
    PRESS_CONF_DIR      = DRIVE_ROOT / 'data/raw/press_conferences'
    SPEECHES_CSV        = DRIVE_ROOT / 'data/raw/all_ECB_speeches.csv'
    OUTPUT_DIR          = DRIVE_ROOT / 'output'
    MEETING_MAPPING     = OUTPUT_DIR / 'ecb_meeting_mapping.json'
    POLICY_DECISIONS    = OUTPUT_DIR / 'ecb_policy_decisions_FULL_1999_2025.csv'
    AGENT_IM_OUT        = OUTPUT_DIR / 'agent_im_scores.json'
    AGENT_IB_OUT        = OUTPUT_DIR / 'agent_ib_scores.json'
    PROB_DIST_OUT       = OUTPUT_DIR / 'probability_distributions.json'
    SURPRISES_OUT       = OUTPUT_DIR / 'surprises_timeseries.csv'
    CHECKPOINTS_DIR     = OUTPUT_DIR / 'checkpoints'
    BLACKOUT_DAYS       = 10
    SPEECH_LOOKBACK     = 28
    GEMINI_API_KEY      = ''
    GEMINI_MODEL        = 'gemini-2.5-flash'
    GEMINI_TEMPERATURE  = 0.1
    GEMINI_MAX_TOKENS   = 4096
    API_DELAY_SEC       = 1.2
    CHUNK_SIZE_CHARS    = 12_000
    CHUNK_OVERLAP_CHARS = 600

cfg = Config()
cfg.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
cfg.CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)

# ── Gemini client  ────────────────────
client = genai.Client(api_key=cfg.GEMINI_API_KEY)

def call_gemini(prompt: str, retries: int = 3) -> str:
    """Call Gemini and return raw text. Retries on failure."""
    for attempt in range(retries):
        try:
            response = client.models.generate_content(
                model=cfg.GEMINI_MODEL,
                contents=prompt,
                config={
                    "temperature": cfg.GEMINI_TEMPERATURE,
                    "max_output_tokens": cfg.GEMINI_MAX_TOKENS,
                }
            )
            time.sleep(cfg.API_DELAY_SEC)
            return response.text
        except Exception as e:
            log.warning(f"Gemini attempt {attempt+1}/{retries} failed: {e}")
            time.sleep(3 * (attempt + 1))
    return ""

def parse_json_response(raw: str) -> Optional[dict]:
    """Extract JSON from Gemini response, handles markdown fences."""
    if not raw:
        return None
    raw = re.sub(r'```(?:json)?', '', raw).strip().strip('`').strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        m = re.search(r'\{.*\}', raw, re.DOTALL)
        if m:
            try:
                return json.loads(m.group())
            except:
                pass
    log.warning("Could not parse JSON from response.")
    return None

# ── Quick API test ────────────────────────────────────────────
try:
    r = call_gemini('Reply with this exact JSON and nothing else: {"status": "ok"}')
    parsed = parse_json_response(r)
    if parsed and parsed.get('status') == 'ok':
        log.info(f"✅ Gemini API connected — model: {cfg.GEMINI_MODEL}")
    else:
        log.warning(f"⚠️ Gemini responded but unexpected: {r[:100]}")
except Exception as e:
    log.error(f"❌ Gemini API error: {e}")

print("\n✅ Cell 1 complete.")

## Cell 2 — Load Data & Verify Inventory

In [ ]:
# ============================================================
# CELL 2 — LOAD MEETING MAPPING & POLICY DECISIONS
# ============================================================

# ── Load meeting mapping ──────────────────────────────────────
with open(cfg.MEETING_MAPPING) as f:
    mapping_list = json.load(f)

# Converti lista → dizionario keyed by meeting_date
meeting_mapping = {m['meeting_date']: m for m in mapping_list}
meetings = sorted(meeting_mapping.keys())
log.info(f"✅ Meeting mapping loaded: {len(meetings)} meetings ({meetings[0]} → {meetings[-1]})")

# ── Load policy decisions ─────────────────────────────────────
policy_df = pd.read_csv(cfg.POLICY_DECISIONS, parse_dates=['meeting_date'])
policy_df['meeting_date_str'] = policy_df['meeting_date'].dt.strftime('%Y-%m-%d')
policy_df.set_index('meeting_date_str', inplace=True)

log.info(f"✅ Policy decisions loaded: {len(policy_df)} rows")
print("\nPolicy decisions columns:", list(policy_df.columns))
print(policy_df[['mro_rate','dfr_rate','mlf_rate']].head())

# ── Load speeches CSV ─────────────────────────────────────────
speeches_df = pd.read_csv(cfg.SPEECHES_CSV, sep='|', parse_dates=['date'],
                           on_bad_lines='skip')
speeches_df['date'] = pd.to_datetime(speeches_df['date'], errors='coerce')
speeches_df.dropna(subset=['date','contents'], inplace=True)
speeches_df.sort_values('date', inplace=True)
log.info(f"✅ Speeches CSV loaded: {len(speeches_df)} speeches")

# ── Count available raw files ─────────────────────────────────
def count_files(folder: Path, ext: str) -> int:
    if not folder.exists(): return 0
    return len(list(folder.glob(f'*.{ext}')))

counts = {
    'Accounts (PDF)':           count_files(cfg.ACCOUNTS_DIR, 'pdf'),
    'Economic Bulletins (PDF)': count_files(cfg.ECON_BULLETINS_DIR, 'pdf'),
    'Monthly Bulletins (PDF)':  count_files(cfg.MONTHLY_BULL_DIR, 'pdf'),
    'Press Conferences (HTML)': count_files(cfg.PRESS_CONF_DIR, 'html'),
}

print("\n📦 File inventory:")
for k, v in counts.items():
    print(f"  {k:30s}: {v:4d}")

print("\n✅ Cell 2 complete.")

## Cell 3 — Document Processing & Temporal Utilities

In [ ]:
# ============================================================
# CELL 3 — TEXT EXTRACTION, CHUNKING, TEMPORAL FILTERING
# ============================================================

# ────────────────────────────────────────────────────────────
# 3A. Dataclass for document chunks
# ────────────────────────────────────────────────────────────
@dataclass
class DocumentChunk:
    doc_id:           str
    doc_type:         str          # 'account' | 'bulletin' | 'speech' | 'press_conf'
    meeting_ref:      str          # ISO date of the TARGET meeting
    publication_date: date
    chunk_index:      int
    total_chunks:     int
    text:             str
    speaker:          Optional[str] = None
    speaker_weight:   Optional[float] = None

    @property
    def word_count(self) -> int:
        return len(self.text.split())


# ────────────────────────────────────────────────────────────
# 3B. PDF text extraction (pdfplumber)
# ────────────────────────────────────────────────────────────
def extract_pdf_text(path: Path) -> str:
    """Extract full text from PDF using pdfplumber. Returns empty string on failure."""
    try:
        with pdfplumber.open(path) as pdf:
            pages = []
            for page in pdf.pages:
                txt = page.extract_text()
                if txt:
                    pages.append(txt)
        raw = "\n\n".join(pages)
        # Clean ECB boilerplate / headers
        raw = re.sub(r'ECB Economic Bulletin,? Issue \d+/\d+\s*', '', raw)
        raw = re.sub(r'ECB Monthly Bulletin\s+\w+ \d{4}\s*', '', raw)
        raw = re.sub(r'\n{3,}', '\n\n', raw)
        return raw.strip()
    except Exception as e:
        log.warning(f"PDF extraction failed for {path.name}: {e}")
        return ""


def extract_html_text(path: Path) -> str:
    """Extract text from ECB press conference HTML."""
    try:
        with open(path, 'r', encoding='utf-8', errors='ignore') as f:
            soup = BeautifulSoup(f.read(), 'html.parser')
        # Remove nav/footer/header tags
        for tag in soup(['nav','footer','header','script','style']):
            tag.decompose()
        text = soup.get_text(separator='\n')
        text = re.sub(r'\n{3,}', '\n\n', text)
        return text.strip()
    except Exception as e:
        log.warning(f"HTML extraction failed for {path.name}: {e}")
        return ""


# ────────────────────────────────────────────────────────────
# 3C. Smart chunking
# ────────────────────────────────────────────────────────────
def smart_chunk(text: str, chunk_size: int = cfg.CHUNK_SIZE_CHARS,
                overlap: int = cfg.CHUNK_OVERLAP_CHARS) -> List[str]:
    """
    Split text into overlapping chunks, breaking at paragraph boundaries.
    """
    paragraphs = text.split('\n\n')
    chunks, current = [], []
    current_len = 0

    for para in paragraphs:
        para = para.strip()
        if not para:
            continue
        if current_len + len(para) > chunk_size and current:
            chunks.append('\n\n'.join(current))
            # keep last paragraph(s) as overlap
            overlap_text = '\n\n'.join(current)[-overlap:]
            current = [overlap_text, para]
            current_len = len(overlap_text) + len(para)
        else:
            current.append(para)
            current_len += len(para)

    if current:
        chunks.append('\n\n'.join(current))

    return [c for c in chunks if len(c.strip()) > 100]


# ────────────────────────────────────────────────────────────
# 3D. Temporal constraint validator
# ────────────────────────────────────────────────────────────
def validate_ex_ante(publication_date: date, meeting_date: date,
                     doc_type: str = 'document') -> bool:
    """
    Returns True ONLY if the document is ex-ante w.r.t. the target meeting.
    Speeches additionally must respect the 10-day blackout.
    """
    if publication_date >= meeting_date:
        log.debug(f"REJECTED (look-ahead): {doc_type} pub={publication_date} >= meeting={meeting_date}")
        return False
    if doc_type == 'speech':
        blackout_start = meeting_date - timedelta(days=cfg.BLACKOUT_DAYS)
        if publication_date >= blackout_start:
            log.debug(f"REJECTED (blackout): speech pub={publication_date} in blackout window")
            return False
    return True


# ────────────────────────────────────────────────────────────
# 3E. Speaker hierarchy weights
# ────────────────────────────────────────────────────────────
SPEAKER_ROLES = {
    # ── PRESIDENTS ──────────────────────────────────────────
    SPEAKER_ROLES = {
    # ── PRESIDENTS ──────────────────────────────────────────
    'Christine Lagarde':       ('president',   date(2019,11, 1), date(2030, 1, 1), 1.00),
    'Mario Draghi':            ('president',   date(2011,11, 1), date(2019,10,31), 1.00),
    'Jean-Claude Trichet':     ('president',   date(2003,11, 1), date(2011,10,31), 1.00),
    'Willem Duisenberg':       ('president',   date(1998, 6, 1), date(2003,10,31), 1.00),
    # ── VICE PRESIDENTS ─────────────────────────────────────
    'Luis de Guindos':         ('vp',          date(2018, 6, 1), date(2030, 1, 1), 0.85),
    'Vitor Constancio':        ('vp',          date(2010, 6, 1), date(2018, 5,31), 0.85),
    'Lucas Papademos':         ('vp',          date(2002,11, 1), date(2010,10,31), 0.85),
    'Christian Noyer':         ('vp',          date(1998, 6, 1), date(2002,10,31), 0.85),
    # ── CHIEF ECONOMISTS ────────────────────────────────────
    'Philip Lane':             ('chief_econ',  date(2019, 6, 1), date(2030, 1, 1), 0.80),
    'Peter Praet':             ('chief_econ',  date(2012, 6, 1), date(2019, 5,31), 0.80),
    'Juergen Stark':           ('chief_econ',  date(2006, 6, 1), date(2012, 1, 1), 0.80),
    'Otmar Issing':            ('chief_econ',  date(1998, 6, 1), date(2006, 5,31), 0.80),
    # ── EXECUTIVE BOARD ─────────────────────────────────────
    'Isabel Schnabel':         ('exec_board',  date(2020, 1, 1), date(2030, 1, 1), 0.70),
    'Frank Elderson':          ('exec_board',  date(2021, 1, 1), date(2030, 1, 1), 0.70),
    'Piero Cipollone':         ('exec_board',  date(2023,11, 1), date(2030, 1, 1), 0.70),
    'Fabio Panetta':           ('exec_board',  date(2020, 1, 1), date(2023,10,31), 0.70),
    'Yves Mersch':             ('exec_board',  date(2012,12, 1), date(2020,12,14), 0.70),
    'Benoît Cœuré':            ('exec_board',  date(2012, 1, 1), date(2019,12,31), 0.70),
    'Sabine Lautenschläger':   ('exec_board',  date(2014, 1, 1), date(2019,10,31), 0.70),
    'Jorg Asmussen':           ('exec_board',  date(2012, 1, 1), date(2014, 1, 1), 0.70),
    'Lorenzo Bini Smaghi':     ('exec_board',  date(2005, 6, 1), date(2011,12,31), 0.70),
    'Gertrude Tumpel-Gugerell': ('exec_board', date(2003, 6, 1), date(2011, 5,31), 0.70),
    'José Manuel González-Páramo': ('exec_board', date(2004,6,1), date(2012,5,31), 0.70),
    'Otmar Issing':            ('exec_board',  date(1998, 6, 1), date(2006, 5,31), 0.70),
    'Eugenio Domingo Solans':  ('exec_board',  date(1998, 6, 1), date(2004, 5,31), 0.70),
    'Sirkka Hämäläinen':       ('exec_board',  date(1998, 6, 1), date(2003, 5,31), 0.70),
    'Tommaso Padoa-Schioppa':  ('exec_board',  date(1998, 6, 1), date(2005,12,31), 0.70),
    # ── NCB GOVERNORS — PAESI GRANDI VERIFICATI ─────────────
    # Germania (Bundesbank)
    'Joachim Nagel':           ('ncb_governor', date(2022, 1, 1), date(2030, 1, 1), 0.55),
    'Jens Weidmann':           ('ncb_governor', date(2011, 5, 1), date(2021,12,31), 0.55),
    'Axel Weber':              ('ncb_governor', date(2004, 5, 1), date(2011, 4,30), 0.55),
    'Ernst Welteke':           ('ncb_governor', date(1999, 9, 1), date(2004, 4,16), 0.55),
    # Francia (Banque de France)
    'François Villeroy de Galhau': ('ncb_governor', date(2015,11,1), date(2030,1,1), 0.55),
    'Christian Noyer':         ('ncb_governor', date(2003,11, 1), date(2015,10,31), 0.55),
    'Jean-Claude Trichet':     ('ncb_governor', date(1993, 9, 1), date(2003,10,31), 0.55),
    # Italia (Banca d'Italia)
    'Fabio Panetta':           ('ncb_governor', date(2023,11, 1), date(2030, 1, 1), 0.55),
    'Ignazio Visco':           ('ncb_governor', date(2011,11, 1), date(2023,10,31), 0.55),
    'Mario Draghi':            ('ncb_governor', date(2006, 1, 1), date(2011,10,31), 0.55),
    'Antonio Fazio':           ('ncb_governor', date(1993, 1, 1), date(2005,12,31), 0.55),
    # Spagna (Banco de España)
    'José Luis Escrivá':       ('ncb_governor', date(2024, 2, 1), date(2030, 1, 1), 0.55),
    'Pablo Hernández de Cos':  ('ncb_governor', date(2018, 6, 1), date(2024, 1,31), 0.55),
    'Luis Linde':              ('ncb_governor', date(2012, 6, 1), date(2018, 5,31), 0.55),
    'Miguel Fernández Ordóñez': ('ncb_governor', date(2006,7,1), date(2012,6,10), 0.55),
    'Jaime Caruana':           ('ncb_governor', date(2000, 7, 1), date(2006, 6,30), 0.55),
    # Olanda (De Nederlandsche Bank)
    'Klaas Knot':              ('ncb_governor', date(2011, 7, 1), date(2030, 1, 1), 0.55),
    'Nout Wellink':            ('ncb_governor', date(1997, 7, 1), date(2011, 6,30), 0.55),
}
DEFAULT_WEIGHTS = {'president': 1.0, 'vp': 0.85, 'chief_econ': 0.80,
                   'exec_board': 0.70, 'ncb_governor': 0.55, 'unknown': 0.40}

def get_speaker_weight(speaker: str, speech_date: date) -> float:
    for name, (role, start, end, weight) in SPEAKER_ROLES.items():
        if name.lower() in speaker.lower() and start <= speech_date <= end:
            return weight
    # Heuristic: NCB governors typically have surname in list
    return DEFAULT_WEIGHTS['unknown']


# ────────────────────────────────────────────────────────────
# 3F. Speech filter for a given target meeting
# ────────────────────────────────────────────────────────────
def filter_speeches(target_meeting_date: date) -> pd.DataFrame:
    """
    Returns speeches in the [lookback_start, blackout_start) window.
    Enforces 28-day lookback and 10-day blackout.
    """
    blackout_start = target_meeting_date - timedelta(days=cfg.BLACKOUT_DAYS)
    lookback_start = target_meeting_date - timedelta(days=cfg.SPEECH_LOOKBACK)

    mask = (
        (speeches_df['date'].dt.date >= lookback_start) &
        (speeches_df['date'].dt.date <  blackout_start)
    )
    filtered = speeches_df[mask].copy().reset_index(drop=True)
    w_list = []
    for _, row in filtered.iterrows():
        speaker = str(row['speaker']) if 'speaker' in row.index and pd.notna(row['speaker']) else ''
        d = row['date'].date() if hasattr(row['date'], 'date') else row['date']
        w_list.append(get_speaker_weight(speaker, d))
    filtered['weight'] = w_list
    return filtered


# ────────────────────────────────────────────────────────────
# 3G. Gemini call wrapper with retry
# ────────────────────────────────────────────────────────────
def call_gemini(prompt: str, retries: int = 3) -> str:
    for attempt in range(retries):
        try:
            response = client.models.generate_content(
                model=cfg.GEMINI_MODEL,
                contents=prompt,
                config={
                    "temperature": cfg.GEMINI_TEMPERATURE,
                    "max_output_tokens": cfg.GEMINI_MAX_TOKENS,
                }
            )
            time.sleep(cfg.API_DELAY_SEC)
            return response.text
        except Exception as e:
            log.warning(f"Gemini attempt {attempt+1}/{retries} failed: {e}")
            time.sleep(3 * (attempt + 1))
    return ""


def parse_json_response(raw: str) -> Optional[dict]:
    """Extract JSON from Gemini response (handles markdown fences)."""
    # Strip markdown code fences
    raw = re.sub(r"```(?:json)?", "", raw).strip().strip("`")
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        # Try to find JSON block
        m = re.search(r'\{.*\}', raw, re.DOTALL)
        if m:
            try:
                return json.loads(m.group())
            except:
                pass
    log.warning("Could not parse JSON from response.")
    return None


print("✅ Cell 3 complete — utilities ready.")


## Cell 4 — Agent IM: Monetary Policy Intelligence

**Input:** ECB Account (meeting t-1) + Press Conference (meeting t-1)  
**Output:** Hawk/dove stance, forward guidance, dissent level, baseline probability distribution

In [ ]:
# ============================================================
# CELL 4 — AGENT IM (POLICY INTELLIGENCE)
# ============================================================

from pydantic import BaseModel, Field, model_validator

class CouncilDivision(BaseModel):
    hawks_fraction:      float = Field(ge=0.0, le=1.0)
    doves_fraction:      float = Field(ge=0.0, le=1.0)
    centrists_fraction:  float = Field(ge=0.0, le=1.0)
    dissent_detected:    bool
    dissent_description: str

class ForwardGuidance(BaseModel):
    type:              str
    verbatim_extract:  str
    implied_direction: str

class BaselineDistribution(BaseModel):
    hike_large:    float = Field(ge=0.0, le=1.0)
    hike_standard: float = Field(ge=0.0, le=1.0)
    hold:          float = Field(ge=0.0, le=1.0)
    cut_standard:  float = Field(ge=0.0, le=1.0)
    cut_large:     float = Field(ge=0.0, le=1.0)

    @model_validator(mode='after')
    def must_sum_to_one(self):
        total = (self.hike_large + self.hike_standard +
                 self.hold + self.cut_standard + self.cut_large)
        if abs(total - 1.0) > 0.01:
            raise ValueError(f"Distribution sums to {total:.3f}, must be 1.0")
        return self

# ── System prompt ─────────────────────────────────────────────
AGENT_IM_SYSTEM = """You are Agent IM (Monetary Intelligence), a specialist in ECB monetary policy.
Your task: analyse the provided ECB Account/Press Conference from meeting t-1 and produce
a structured JSON assessment of the Governing Council's monetary stance.

CRITICAL RULES:
- Temporal anchor: you only have information up to the ACCOUNT PUBLICATION DATE provided.
- Never infer information from data released after that date.
- All probability fields must sum to exactly 1.0.
- Return ONLY valid JSON — no markdown, no preamble, no explanation outside JSON.

OUTPUT JSON SCHEMA (all fields required):
{{
  "meeting_analyzed": "YYYY-MM-DD",
  "account_pub_date": "YYYY-MM-DD",
  "target_meeting": "YYYY-MM-DD",
  "hawkishness_score": float [-1.0 very dovish to +1.0 very hawkish],
  "hawkishness_confidence": float [0.0-1.0],
  "hawkishness_rationale": "concise rationale (max 150 words)",
  "council_division": {{
    "hawks_fraction": float,
    "doves_fraction": float,
    "centrists_fraction": float,
    "dissent_detected": bool,
    "dissent_description": "description or None detected"
  }},
  "forward_guidance": {{
    "type": "Odyssean|Delphic|None",
    "verbatim_extract": "exact quote or None",
    "implied_direction": "hike|hold|cut|ambiguous"
  }},
  "uncertainty_score": float [0=certain to 1=maximum uncertainty],
  "baseline_distribution": {{
    "hike_large": float,
    "hike_standard": float,
    "hold": float,
    "cut_standard": float,
    "cut_large": float
  }},
  "shock_indicators": ["up to 3 phrases signalling potential surprise"],
  "nonstandard_tools": ["APP", "PEPP", "TPI", etc. if mentioned],
  "key_phrases": ["up to 5 most policy-relevant phrases verbatim"]
}}"""

# ── Merge prompt ──────────────────────────────────────────────
AGENT_IM_MERGE_PROMPT = """You are Agent IM. You received {n} chunk-level analyses of the same ECB Account.
Merge them into ONE coherent JSON object following the same schema.
Average numeric scores; take the union of lists; resolve contradictions by majority/weight.
Return ONLY valid JSON.

CHUNK ANALYSES:
{chunk_jsons}"""


def run_agent_im(
    meeting_date_str: str,
    account_path: Optional[Path],
    press_conf_path: Optional[Path],
    prev_meeting_date_str: str,
    account_pub_date_str: str,
) -> Optional[dict]:
    texts = []

    if account_path and account_path.exists():
        txt = extract_pdf_text(account_path)
        if len(txt) > 200:
            texts.append(('ECB Account', txt))
        else:
            log.warning(f"Account PDF extracted very little text: {account_path.name}")

    if press_conf_path and press_conf_path.exists():
        txt = extract_html_text(press_conf_path)
        if len(txt) > 200:
            texts.append(('Press Conference', txt))

    if not texts:
        log.warning(f"Agent IM: no usable text for {meeting_date_str}")
        return None

    combined = "\n\n---\n\n".join(f"[{t}]\n{c}" for t, c in texts)
    chunks = smart_chunk(combined)
    log.info(f"Agent IM: processing {len(chunks)} chunks for {meeting_date_str}")

    chunk_results = []
    for i, chunk in enumerate(chunks):
        prompt = AGENT_IM_SYSTEM + f"""

=== TEMPORAL ANCHOR ===
Account publication date: {account_pub_date_str}
Meeting analyzed (t-1):   {prev_meeting_date_str}
Target meeting (t):       {meeting_date_str}

=== DOCUMENT ===
Type: ECB Account + Press Conference
[Chunk {i+1}/{len(chunks)}]

{chunk}

=== TASK ===
Analyse the Governing Council debate above. Return ONLY the JSON object.
"""
        raw = call_gemini(prompt)
        parsed = parse_json_response(raw)
        if parsed:
            chunk_results.append(parsed)
        else:
            log.warning(f"Chunk {i+1} returned no valid JSON")

    if not chunk_results:
        return None

    if len(chunk_results) == 1:
        return chunk_results[0]

    # Merge multiple chunks
    merge_prompt = (AGENT_IM_SYSTEM + "\n\n" +
                    AGENT_IM_MERGE_PROMPT.format(
                        n=len(chunk_results),
                        chunk_jsons=json.dumps(chunk_results, indent=2)
                    ))
    merged_raw = call_gemini(merge_prompt)
    return parse_json_response(merged_raw) or chunk_results[0]


def test_agent_im_single(meeting_date_str: str = "2024-06-06"):
    """Test Agent IM on one meeting using exact filenames from meeting_mapping."""
    info = meeting_mapping.get(meeting_date_str, {})
    if not info:
        log.error(f"Meeting {meeting_date_str} not found in mapping")
        return None

    # ── Account: usa il filename esatto dal mapping ────────────
    account_filename = info.get('account_previous')
    account_file = cfg.ACCOUNTS_DIR / account_filename if account_filename else None
    if account_file and not account_file.exists():
        log.warning(f"Account file not found: {account_file.name}")
        account_file = None

    # ── Press conference: meeting precedente per indice ────────
    meeting_index = info.get('meeting_index', 0)
    prev_meeting = None
    for d, m in meeting_mapping.items():
        if m.get('meeting_index') == meeting_index - 1:
            prev_meeting = d
            break

    pc_file = None
    if prev_meeting:
        pc_file = cfg.PRESS_CONF_DIR / f"pressconf_{prev_meeting.replace('-','')}.html"
        if not pc_file.exists():
            log.warning(f"Press conf not found: {pc_file.name}")
            pc_file = None

    account_pub_date = info.get('account_pub_date', '')

    log.info(f"Target meeting:   {meeting_date_str}")
    log.info(f"Previous meeting: {prev_meeting}")
    log.info(f"Account file:     {account_file.name if account_file else 'NOT FOUND'}")
    log.info(f"Press conf file:  {pc_file.name if pc_file else 'NOT FOUND'}")
    log.info(f"Account pub date: {account_pub_date}")

    result = run_agent_im(
        meeting_date_str=meeting_date_str,
        account_path=account_file,
        press_conf_path=pc_file,
        prev_meeting_date_str=prev_meeting or '',
        account_pub_date_str=account_pub_date,
    )
    print(json.dumps(result, indent=2))
    return result

print("✅ Cell 4 ready — run test_agent_im_single('2024-06-06') to test.")

## Cell 5 — Agent IB: Economic Conditions

**Input:** Economic Bulletin (most recent pre-blackout) + filtered speeches (28-day lookback, 10-day blackout)  
**Output:** Economic condition scores with HICP inflation as primary focus (ECB mandate)

In [ ]:
# ============================================================
# CELL 5 — AGENT IB (ECONOMIC CONDITIONS)
# ============================================================

class AgentIBChunkOutput(BaseModel):
    inflation_outlook:   float = Field(ge=-1.0, le=1.0)
    growth_prospects:    float = Field(ge=-1.0, le=1.0)
    labor_market:        float = Field(ge=-1.0, le=1.0)
    policy_stance:       float = Field(ge=-1.0, le=1.0)
    inflation_rationale: str
    growth_rationale:    str
    labor_rationale:     str
    policy_rationale:    str
    confidence:          float = Field(ge=0.0, le=1.0)

class AgentIBOutput(BaseModel):
    target_meeting:       str
    bulletin_date:        str
    inflation_outlook:    float = Field(ge=-1.0, le=1.0)
    growth_prospects:     float = Field(ge=-1.0, le=1.0)
    labor_market:         float = Field(ge=-1.0, le=1.0)
    policy_stance:        float = Field(ge=-1.0, le=1.0)
    speech_hawkishness:   float = Field(ge=-1.0, le=1.0)
    speech_n_processed:   int
    aggregate_econ_index: float
    rationales:           dict
    confidence:           float = Field(ge=0.0, le=1.0)


AGENT_IB_SYSTEM = """You are Agent IB (Economic Conditions), a specialist in ECB economic analysis.
Your task: read the provided ECB Economic Bulletin / Monthly Bulletin excerpt and extract
quantitative economic condition scores that the Governing Council would use for rate decisions.

ECB MANDATE PRIORITY:
- Price stability (HICP inflation) is the PRIMARY mandate -> weight it most heavily.
- Score -1.0 = extremely dovish/weak conditions (far below target/trend)
- Score +1.0 = extremely hawkish/strong conditions (far above target/trend)
- Score  0.0 = neutral / on target

CRITICAL RULES:
- Only use information from the document text provided.
- Temporal anchor: you only know what was available as of the BULLETIN PUBLICATION DATE.
- Return ONLY valid JSON — no markdown, no explanation outside JSON.

OUTPUT JSON SCHEMA:
{{
  "inflation_outlook": float [-1.0 to +1.0],
  "growth_prospects":  float [-1.0 to +1.0],
  "labor_market":      float [-1.0 to +1.0],
  "policy_stance":     float [-1.0 to +1.0],
  "inflation_rationale":  "key HICP/inflation evidence cited (max 100 words)",
  "growth_rationale":     "key GDP/activity evidence cited (max 80 words)",
  "labor_rationale":      "key employment evidence cited (max 80 words)",
  "policy_rationale":     "explicit rate/tool signals mentioned (max 80 words)",
  "confidence": float [0.0-1.0]
}}"""

AGENT_IB_SPEECH_SYSTEM = """You are Agent IB analysing a single ECB speech.
Score the speaker's economic assessment on [-1.0, +1.0] scales.
Focus especially on inflation/price stability signals.
Return ONLY valid JSON matching this schema:
{{
  "inflation_outlook": float,
  "growth_prospects":  float,
  "labor_market":      float,
  "policy_stance":     float,
  "confidence":        float,
  "key_signals":       ["up to 3 key phrases"]
}}"""


def run_agent_ib_bulletin(
    bulletin_path: Path,
    bulletin_pub_date_str: str,
    target_meeting_str: str,
    doc_type: str = "ECB Economic Bulletin",
) -> Optional[dict]:
    """Process one bulletin through Agent IB, aggregating across chunks."""
    txt = extract_pdf_text(bulletin_path)
    if len(txt) < 500:
        log.warning(f"Agent IB: bulletin too short ({len(txt)} chars): {bulletin_path.name}")
        return None

    chunks = smart_chunk(txt)
    log.info(f"Agent IB: processing {len(chunks)} chunks for {target_meeting_str}")
    chunk_results = []

    for i, chunk in enumerate(chunks):
        prompt = AGENT_IB_SYSTEM + f"""

=== TEMPORAL ANCHOR ===
Bulletin publication date: {bulletin_pub_date_str}
Target meeting:            {target_meeting_str}

=== DOCUMENT: {doc_type} ===
[Chunk {i+1}/{len(chunks)}]

{chunk}

=== TASK ===
Extract economic condition scores. Focus on HICP inflation above all else.
Return ONLY valid JSON.
"""
        raw = call_gemini(prompt)
        parsed = parse_json_response(raw)
        if parsed:
            chunk_results.append(parsed)
        else:
            log.warning(f"Agent IB chunk {i+1} returned no valid JSON")

    if not chunk_results:
        return None

    # Average numeric scores across chunks
    numeric_keys = ['inflation_outlook', 'growth_prospects', 'labor_market',
                    'policy_stance', 'confidence']
    aggregated = {}
    for k in numeric_keys:
        vals = [float(r[k]) for r in chunk_results
                if k in r and r[k] is not None and isinstance(r[k], (int, float))]
        aggregated[k] = float(np.mean(vals)) if vals else 0.0

    # Text rationales from first chunk
    for k in ['inflation_rationale', 'growth_rationale', 'labor_rationale', 'policy_rationale']:
        aggregated[k] = chunk_results[0].get(k, '')

    return aggregated


def run_agent_ib_speeches(target_meeting_date: date) -> dict:
    """
    Aggregate speech signals for a target meeting.
    Applies 28-day lookback and 10-day blackout automatically.
    """
    filtered = filter_speeches(target_meeting_date)
    if filtered.empty:
        return {"speech_hawkishness": 0.0, "speech_n_processed": 0}

    filtered = filtered.sort_values('weight', ascending=False).head(15)
    weighted_scores = []
    weights = []

    for _, row in filtered.iterrows():
        speech_text = str(row.get('contents', ''))
        if len(speech_text) < 300:
            continue
        speech_text = speech_text[:8_000]

        prompt = AGENT_IB_SPEECH_SYSTEM + f"""

Speech date: {row['date'].date()}
Speaker: {row.get('speaker', 'Unknown')}
Title: {row.get('title', '')}

TEXT:
{speech_text}

Return ONLY valid JSON."""

        raw = call_gemini(prompt)
        parsed = parse_json_response(raw)
        if parsed:
            hawk = (0.5 * parsed.get('inflation_outlook', 0) +
                    0.3 * parsed.get('growth_prospects', 0) +
                    0.2 * parsed.get('labor_market', 0))
            w = float(row['weight'])
            weighted_scores.append(hawk * w)
            weights.append(w)

    if not weights:
        return {"speech_hawkishness": 0.0, "speech_n_processed": 0}

    agg_hawk = float(np.sum(weighted_scores) / np.sum(weights))
    return {
        "speech_hawkishness": round(agg_hawk, 4),
        "speech_n_processed": len(weights),
    }


def run_agent_ib(
    target_meeting_str: str,
    bulletin_path: Optional[Path],
    bulletin_pub_date_str: str,
    doc_type: str = "ECB Economic Bulletin",
) -> Optional[dict]:
    """Full Agent IB run for one meeting."""
    target_meeting_date = date.fromisoformat(target_meeting_str)

    bulletin_scores = {}
    if bulletin_path and bulletin_path.exists():
        bulletin_scores = run_agent_ib_bulletin(
            bulletin_path, bulletin_pub_date_str, target_meeting_str, doc_type
        ) or {}

    speech_scores = run_agent_ib_speeches(target_meeting_date)

    if not bulletin_scores:
        log.warning(f"Agent IB: no bulletin scores for {target_meeting_str}")
        return None

    infl  = bulletin_scores.get('inflation_outlook', 0.0)
    growt = bulletin_scores.get('growth_prospects',  0.0)
    labor = bulletin_scores.get('labor_market',      0.0)
    agg_econ = 0.5 * infl + 0.3 * growt + 0.2 * labor

    return {
        "target_meeting":       target_meeting_str,
        "bulletin_date":        bulletin_pub_date_str,
        "inflation_outlook":    infl,
        "growth_prospects":     growt,
        "labor_market":         labor,
        "policy_stance":        bulletin_scores.get('policy_stance', 0.0),
        "speech_hawkishness":   speech_scores["speech_hawkishness"],
        "speech_n_processed":   speech_scores["speech_n_processed"],
        "aggregate_econ_index": round(agg_econ, 4),
        "rationales": {
            k: bulletin_scores.get(k, '') for k in
            ['inflation_rationale', 'growth_rationale', 'labor_rationale', 'policy_rationale']
        },
        "confidence": bulletin_scores.get('confidence', 0.5),
    }

print("✅ Cell 5 ready — Agent IB functions defined.")

## Cell 6 — Agent II: Synthesis & Probability Distribution

**Input:** Agent IM output (hawkishness + baseline distribution) + Agent IB output (economic conditions)  
**Output:** Posterior probability distributions for MRO, DFR, MLF decisions

In [ ]:
# ============================================================
# CELL 6 — AGENT II (SYNTHESIS)
# ============================================================
import scipy.special  # for softmax

# Rate bins with bp values
RATE_BINS = ["hike_large","hike_standard","hold","cut_standard","cut_large"]
BIN_BP     = {"hike_large": 50, "hike_standard": 25, "hold": 0,
              "cut_standard": -25, "cut_large": -50}


def softmax(x: np.ndarray) -> np.ndarray:
    """Numerically stable softmax."""
    e = np.exp(x - np.max(x))
    return e / e.sum()


def compute_distribution_moments(dist: dict) -> dict:
    """
    Compute E[Δi], Var[Δi], Shannon entropy, modal outcome.
    dist = {bin: probability}
    """
    probs = np.array([dist[b] for b in RATE_BINS])
    bps   = np.array([BIN_BP[b] for b in RATE_BINS])

    e_change  = float(np.dot(probs, bps))
    variance  = float(np.dot(probs, (bps - e_change)**2))
    entropy   = float(-np.sum(probs * np.log(probs + 1e-12)))
    modal_bin = RATE_BINS[int(np.argmax(probs))]

    return {
        "expected_change_bp": round(e_change, 4),
        "variance_bp2":       round(variance, 4),
        "entropy":            round(entropy, 4),
        "modal_outcome":      modal_bin,
    }


def run_agent_ii(
    im_output: dict,
    ib_output: dict,
    target_meeting_str: str,
) -> Optional[dict]:
    """
    Bayesian-style update: start from IM baseline, update with IB economic signal.

    Policy signal  = 0.6*hawkishness_IM + 0.2*FG_implied + 0.2*IB_policy_stance
    Economic signal = 0.5*inflation + 0.3*growth + 0.2*labor  (already in ib aggregate)
    Combined signal = 0.55*policy_signal + 0.45*econ_signal

    The combined signal shifts the log-odds of the IM baseline distribution,
    then we renormalize (softmax).
    """
    # ── Compute signals ───────────────────────────────────────
    hawk_im = float(im_output.get('hawkishness_score', 0.0))

    # Forward guidance implied direction → numeric
    fg = im_output.get('forward_guidance', {})
    fg_dir_map = {'hike': +0.6, 'hold': 0.0, 'cut': -0.6, 'ambiguous': 0.0}
    fg_signal = fg_dir_map.get(fg.get('implied_direction', 'ambiguous'), 0.0)

    ib_policy = float(ib_output.get('policy_stance', 0.0))
    econ_agg  = float(ib_output.get('aggregate_econ_index', 0.0))
    speech_h  = float(ib_output.get('speech_hawkishness', 0.0))

    # Blend speech signal into economic index
    econ_with_speech = 0.7 * econ_agg + 0.3 * speech_h

    policy_signal   = 0.6*hawk_im + 0.2*fg_signal + 0.2*ib_policy
    combined_signal = 0.55*policy_signal + 0.45*econ_with_speech

    # ── Update IM baseline ────────────────────────────────────
    baseline = im_output.get('baseline_distribution', {})
    prior_probs = np.array([
        float(baseline.get(b, 0.2)) for b in RATE_BINS
    ])
    prior_probs = prior_probs / prior_probs.sum()   # ensure normalised

    # Shift log-odds proportionally to combined_signal
    # Positive signal → shift probability mass toward hike bins
    shift_weights = np.array([2.0, 1.0, 0.0, -1.0, -2.0])  # per bin
    log_prior = np.log(prior_probs + 1e-12)
    log_posterior = log_prior + combined_signal * shift_weights
    posterior_probs = softmax(log_posterior)

    posterior = {b: round(float(p), 4) for b, p in zip(RATE_BINS, posterior_probs)}
    moments   = compute_distribution_moments(posterior)

    # Hawkishness index from posterior E[Δi] (normalised to [-1, 1])
    hawk_index = float(np.clip(moments['expected_change_bp'] / 50.0, -1.0, 1.0))

    signals_aligned = (policy_signal * econ_with_speech > 0) or (
        abs(policy_signal) < 0.1 or abs(econ_with_speech) < 0.1
    )

    return {
        "target_meeting": target_meeting_str,
        "signal_reconciliation": {
            "policy_signal":    round(policy_signal, 4),
            "econ_signal":      round(econ_with_speech, 4),
            "combined_signal":  round(combined_signal, 4),
            "signals_aligned":  signals_aligned,
        },
        "prior_distribution":    {b: round(float(p), 4) for b, p in zip(RATE_BINS, prior_probs)},
        "posterior_distribution":posterior,
        "distribution_moments":  moments,
        "hawkishness_index":     round(hawk_index, 4),
        "policy_uncertainty_index": round(moments['entropy'] / np.log(len(RATE_BINS)), 4),
        "confidence_overall":    round(float(
            0.5*(im_output.get('hawkishness_confidence', 0.5) +
                 ib_output.get('confidence', 0.5))), 4),
    }


print("✅ Cell 6 ready — Agent II (synthesis) defined.")


## Cell 7 — Agent III: Surprise Calculation

**Formula:** $s_t = \Delta i_t^{\text{actual}} - \mathbb{E}[\Delta i_t | \mathcal{B}_t]$  
**Three measures:** Mechanical, Salience (info-theoretic), Normalised

In [ ]:
# ============================================================
# CELL 7 — AGENT III (SURPRISE CALCULATION) + FULL PIPELINE
# ============================================================

ACTUAL_TO_BIN_MAP = {
    50: "hike_large", 25: "hike_standard", 0: "hold",
    -25: "cut_standard", -50: "cut_large",
    75: "hike_large", 100: "hike_large",
    -75: "cut_large", -100: "cut_large",
}


def map_actual_to_bin(change_bp: float) -> str:
    """Map actual rate change in bp to a distribution bin."""
    rounded = int(round(change_bp / 25.0) * 25)
    if rounded >= 50:  return "hike_large"
    if rounded == 25:  return "hike_standard"
    if rounded == 0:   return "hold"
    if rounded == -25: return "cut_standard"
    return "cut_large"


def compute_surprises(
    ii_output: dict,
    actual_dfr_bp: float,
    actual_mro_bp: float,
    actual_mlf_bp: float,
    rolling_std: Optional[float] = None,
) -> dict:
    """
    Three surprise measures for all three ECB rates.
    Mechanical:  s_mech = actual_bp - E[delta_i_bp]
    Salience:    s_sal  = -log P(actual bin)
    Normalised:  s_norm = s_mech / rolling_std (if available)
    """
    dist     = ii_output['posterior_distribution']
    e_change = ii_output['distribution_moments']['expected_change_bp']

    def _surprises_for_rate(actual_bp: float) -> dict:
        actual_bin = map_actual_to_bin(actual_bp)
        p_actual   = float(dist.get(actual_bin, 1e-6))
        p_actual   = max(p_actual, 1e-6)
        s_mech = actual_bp - e_change
        s_sal  = -np.log(p_actual)
        s_norm = s_mech / rolling_std if (rolling_std and rolling_std > 0) else None
        return {
            "actual_bp":    actual_bp,
            "expected_bp":  round(e_change, 4),
            "actual_bin":   actual_bin,
            "p_actual_bin": round(p_actual, 4),
            "surprise_mech": round(s_mech, 4),
            "surprise_sal":  round(s_sal, 4),
            "surprise_norm": round(s_norm, 4) if s_norm is not None else None,
            "direction": ("hawkish_surprise" if s_mech > 5
                          else "dovish_surprise" if s_mech < -5
                          else "no_surprise"),
        }

    return {
        "meeting_date":  ii_output['target_meeting'],
        "dfr_surprise":  _surprises_for_rate(actual_dfr_bp),
        "mro_surprise":  _surprises_for_rate(actual_mro_bp),
        "mlf_surprise":  _surprises_for_rate(actual_mlf_bp),
        "entropy":       ii_output['distribution_moments']['entropy'],
        "modal_outcome": ii_output['distribution_moments']['modal_outcome'],
    }


def _resolve_paths_for_meeting(mtg: str) -> dict:
    """
    Resolve all file paths for a given meeting date using the mapping.
    Returns a dict with: account_path, pc_path, bulletin_path,
                         bulletin_date_str, doc_type_str, acc_pub, prev
    """
    info = meeting_mapping.get(mtg, {})

    # ── Account: usa filename esatto dal mapping ──────────────
    account_filename = info.get('account_previous')
    account_path = cfg.ACCOUNTS_DIR / account_filename if account_filename else None
    if account_path and not account_path.exists():
        log.warning(f"Account file not found: {account_filename}")
        account_path = None

    # ── Press conference: meeting precedente per indice ───────
    meeting_index = info.get('meeting_index', 0)
    prev = None
    for d, m in meeting_mapping.items():
        if m.get('meeting_index') == meeting_index - 1:
            prev = d
            break

    pc_path = None
    if prev:
        pc_candidate = cfg.PRESS_CONF_DIR / f"pressconf_{prev.replace('-','')}.html"
        if pc_candidate.exists():
            pc_path = pc_candidate

    # ── Bulletin: usa filename esatto dal mapping ─────────────
    bulletin_filename = info.get('bulletin')
    bulletin_date_str = info.get('bulletin_pub_date', mtg)
    year = int(mtg[:4])
    doc_type_str = "ECB Economic Bulletin" if year >= 2015 else "ECB Monthly Bulletin"

    bulletin_path = None
    if bulletin_filename:
        folder = cfg.ECON_BULLETINS_DIR if year >= 2015 else cfg.MONTHLY_BULL_DIR
        candidate = folder / bulletin_filename
        if candidate.exists():
            bulletin_path = candidate
        else:
            log.warning(f"Bulletin file not found: {bulletin_filename}")

    # ── Account publication date ──────────────────────────────
    acc_pub = info.get('account_pub_date', mtg)

    return {
        "account_path":      account_path,
        "pc_path":           pc_path,
        "bulletin_path":     bulletin_path,
        "bulletin_date_str": bulletin_date_str,
        "doc_type_str":      doc_type_str,
        "acc_pub":           acc_pub,
        "prev":              prev or '',
    }


def run_full_pipeline(
    meeting_dates: List[str],
    start_from: int = 0,
) -> List[dict]:
    """
    Run the complete 4-agent pipeline for every meeting in meeting_dates.
    Saves a checkpoint JSON after each meeting.
    Skips meetings already processed (checkpoint exists).
    """
    results = []

    # Load existing checkpoints
    done_dates = set()
    for cp in cfg.CHECKPOINTS_DIR.glob("*.json"):
        done_dates.add(cp.stem)

    for idx, mtg in enumerate(meeting_dates[start_from:], start=start_from):
        if mtg in done_dates:
            cp_path = cfg.CHECKPOINTS_DIR / f"{mtg}.json"
            with open(cp_path) as f:
                results.append(json.load(f))
            log.info(f"[{idx+1}/{len(meeting_dates)}] {mtg} — loaded from checkpoint")
            continue

        log.info(f"[{idx+1}/{len(meeting_dates)}] Processing {mtg}...")

        # ── Resolve file paths ────────────────────────────────
        paths = _resolve_paths_for_meeting(mtg)

        # ── Agent IM ──────────────────────────────────────────
        im_out = run_agent_im(
            meeting_date_str=mtg,
            account_path=paths['account_path'],
            press_conf_path=paths['pc_path'],
            prev_meeting_date_str=paths['prev'],
            account_pub_date_str=paths['acc_pub'],
        )
        if not im_out:
            log.warning(f"Agent IM returned None for {mtg} — skipping.")
            continue

        # ── Agent IB ──────────────────────────────────────────
        ib_out = run_agent_ib(
            target_meeting_str=mtg,
            bulletin_path=paths['bulletin_path'],
            bulletin_pub_date_str=paths['bulletin_date_str'],
            doc_type=paths['doc_type_str'],
        )
        if not ib_out:
            log.warning(f"Agent IB returned None for {mtg} — skipping.")
            continue

        # ── Agent II ──────────────────────────────────────────
        ii_out = run_agent_ii(im_out, ib_out, mtg)
        if not ii_out:
            continue

        # ── Agent III ─────────────────────────────────────────
        if mtg not in policy_df.index:
            log.warning(f"No policy decision for {mtg}")
            continue

        row = policy_df.loc[mtg]

        # Rolling std over past 12 mechanical surprises
        rolling_std = None
        if len(results) >= 6:
            past_mech = [r['surprises']['dfr_surprise']['surprise_mech']
                         for r in results[-12:] if 'surprises' in r]
            if past_mech:
                rolling_std = float(np.std(past_mech))

        surprises = compute_surprises(
            ii_output=ii_out,
            actual_dfr_bp=float(row.get('dfr_change_bp', 0)),
            actual_mro_bp=float(row.get('mro_change_bp', 0)),
            actual_mlf_bp=float(row.get('mlf_change_bp', 0)),
            rolling_std=rolling_std,
        )

        record = {
            "meeting_date": mtg,
            "agent_im":     im_out,
            "agent_ib":     ib_out,
            "agent_ii":     ii_out,
            "surprises":    surprises,
        }
        results.append(record)

        # Save checkpoint
        cp_path = cfg.CHECKPOINTS_DIR / f"{mtg}.json"
        with open(cp_path, 'w') as f:
            json.dump(record, f, indent=2)

        log.info(f"  ✅ {mtg} — DFR surprise: "
                 f"{surprises['dfr_surprise']['surprise_mech']:.1f}bp "
                 f"({surprises['dfr_surprise']['direction']})")

    return results


print("✅ Cell 7 ready — Agent III + full pipeline defined.")


## Cell 8 — Validation, Export & Summary Statistics

In [ ]:
# ============================================================
# CELL 8 — RUN PIPELINE, VALIDATE, EXPORT
# ============================================================

# ── 8A. Single meeting test (proof of concept) ───────────────
TEST_MEETING = "2024-06-06"
print(f"=" * 60)
print(f"🧪 PROOF OF CONCEPT — Meeting {TEST_MEETING}")
print(f"Expected: dovish surprise (MRO cut 4.50%→4.25%)")
print(f"=" * 60)

test_results = run_full_pipeline([TEST_MEETING])

if test_results:
    r = test_results[0]
    print("\n📊 AGENT IM:")
    print(f"  Hawkishness:  {r['agent_im']['hawkishness_score']:+.2f}")
    print(f"  FG type:      {r['agent_im']['forward_guidance']['type']}")
    print(f"  FG direction: {r['agent_im']['forward_guidance']['implied_direction']}")

    print("\n📊 AGENT IB:")
    print(f"  Inflation:    {r['agent_ib']['inflation_outlook']:+.2f}")
    print(f"  Growth:       {r['agent_ib']['growth_prospects']:+.2f}")
    print(f"  Labor:        {r['agent_ib']['labor_market']:+.2f}")
    print(f"  Agg. Econ:    {r['agent_ib']['aggregate_econ_index']:+.2f}")
    print(f"  Speeches:     {r['agent_ib']['speech_n_processed']} processed")

    print("\n📊 AGENT II (posterior distribution):")
    for bin_name, prob in r['agent_ii']['posterior_distribution'].items():
        bar = "█" * int(prob * 30)
        print(f"  {bin_name:16s}: {prob:.3f} {bar}")
    print(f"  E[Δi] = {r['agent_ii']['distribution_moments']['expected_change_bp']:+.1f} bp")
    print(f"  Entropy = {r['agent_ii']['distribution_moments']['entropy']:.3f}")

    print("\n📊 AGENT III (surprises):")
    for rate in ['dfr','mro','mlf']:
        s = r['surprises'][f'{rate}_surprise']
        print(f"  {rate.upper()}: actual={s['actual_bp']:+.0f}bp  "
              f"expected={s['expected_bp']:+.1f}bp  "
              f"surprise={s['surprise_mech']:+.1f}bp  [{s['direction']}]")

# ── 8B. Full pipeline run (all meetings) ─────────────────────
# Uncomment to run the full pipeline across all meetings:
POST_2015_MEETINGS = [m for m in meetings if m >= '2015-01-01']
all_results = run_full_pipeline(POST_2015_MEETINGS)

# ── 8C. Build summary CSV ─────────────────────────────────────
def results_to_dataframe(results: list) -> pd.DataFrame:
    rows = []
    for r in results:
        im   = r.get('agent_im', {})
        ib   = r.get('agent_ib', {})
        ii   = r.get('agent_ii', {})
        surp = r.get('surprises', {})
        dfr  = surp.get('dfr_surprise', {})
        mro  = surp.get('mro_surprise', {})

        row = {
            'meeting_date':          r['meeting_date'],
            # IM scores
            'hawkishness_im':        im.get('hawkishness_score'),
            'uncertainty_im':        im.get('uncertainty_score'),
            'dissent_detected':      im.get('council_division', {}).get('dissent_detected'),
            'fg_type':               im.get('forward_guidance', {}).get('type'),
            'fg_direction':          im.get('forward_guidance', {}).get('implied_direction'),
            # IB scores
            'inflation_outlook':     ib.get('inflation_outlook'),
            'growth_prospects':      ib.get('growth_prospects'),
            'labor_market':          ib.get('labor_market'),
            'speech_hawkishness':    ib.get('speech_hawkishness'),
            'agg_econ_index':        ib.get('aggregate_econ_index'),
            # II outputs
            'expected_change_bp':    ii.get('distribution_moments', {}).get('expected_change_bp'),
            'entropy':               ii.get('distribution_moments', {}).get('entropy'),
            'modal_outcome':         ii.get('distribution_moments', {}).get('modal_outcome'),
            'hawkishness_index_ii':  ii.get('hawkishness_index'),
            'policy_uncertainty':    ii.get('policy_uncertainty_index'),
            'p_hold':                ii.get('posterior_distribution', {}).get('hold'),
            # III surprises
            'dfr_actual_bp':         dfr.get('actual_bp'),
            'dfr_surprise_mech':     dfr.get('surprise_mech'),
            'dfr_surprise_sal':      dfr.get('surprise_sal'),
            'dfr_surprise_norm':     dfr.get('surprise_norm'),
            'dfr_direction':         dfr.get('direction'),
            'mro_surprise_mech':     mro.get('surprise_mech'),
        }
        rows.append(row)

    df = pd.DataFrame(rows)
    df['meeting_date'] = pd.to_datetime(df['meeting_date'])
    df.sort_values('meeting_date', inplace=True)
    df.reset_index(drop=True, inplace=True)
    return df

if test_results:
    df_test = results_to_dataframe(test_results)
    print("\n📋 Summary DataFrame shape:", df_test.shape)
    print(df_test[['meeting_date','dfr_actual_bp','dfr_surprise_mech','dfr_direction']].T)

# ── 8D. Export ────────────────────────────────────────────────
def export_results(results: list, df: pd.DataFrame):
    # CSV timeseries
    csv_path = cfg.SURPRISES_OUT
    df.to_csv(csv_path, index=False)
    log.info(f"✅ Surprises CSV saved: {csv_path}")

    # Full JSON
    json_path = cfg.OUTPUT_DIR / 'ecb_surprises_metadata.json'
    with open(json_path, 'w') as f:
        json.dump(results, f, indent=2, default=str)
    log.info(f"✅ Full JSON saved: {json_path}")

    # Summary stats
    print("\n📈 SUMMARY STATISTICS — DFR Surprises:")
    print(df['dfr_surprise_mech'].describe().round(3))
    n_hawk = (df['dfr_direction'] == 'hawkish_surprise').sum()
    n_dove = (df['dfr_direction'] == 'dovish_surprise').sum()
    n_none = (df['dfr_direction'] == 'no_surprise').sum()
    print(f"  Hawkish surprises: {n_hawk}")
    print(f"  Dovish surprises:  {n_dove}")
    print(f"  No surprise:       {n_none}")

# Uncomment after full run:
# export_results(all_results, results_to_dataframe(all_results))

print("\n✅ Cell 8 complete — pipeline ready.")
print("\nNext steps:")
print("  1. Run test_agent_im_single() to verify Agent IM in isolation")
print("  2. Uncomment full pipeline run in 8B for all 2015–2025 meetings")
print("  3. Extend to 1999–2014 using Monthly Bulletins + Press Conferences")


In [ ]:
def run_pipeline_pre2015(meeting_dates: list) -> list:
    """
    Pipeline per il periodo 1999-2014.
    Agent IM: Press Conference del meeting precedente (no Accounts)
    Agent IB: Monthly Bulletin più recente pre-blackout
    """
    results = []
    done_dates = set(cp.stem for cp in cfg.CHECKPOINTS_DIR.glob("*.json"))

    for idx, mtg in enumerate(meeting_dates):
        if mtg in done_dates:
            cp_path = cfg.CHECKPOINTS_DIR / f"{mtg}.json"
            with open(cp_path) as f:
                results.append(json.load(f))
            log.info(f"[{idx+1}/{len(meeting_dates)}] {mtg} — loaded from checkpoint")
            continue

        log.info(f"[{idx+1}/{len(meeting_dates)}] Processing {mtg}...")

        mtg_date = date.fromisoformat(mtg)
        blackout  = mtg_date - timedelta(days=cfg.BLACKOUT_DAYS)

        # ── Agent IM: Press Conference del meeting precedente ──
        # Trova la press conference più recente prima di questo meeting
        all_pc = sorted(cfg.PRESS_CONF_DIR.glob("pressconf_*.html"))
        prev_pc = None
        prev_pc_date = None
        for pc in reversed(all_pc):
            m = re.search(r'pressconf_(\d{8})\.html', pc.name)
            if m:
                pc_date = date(int(m.group(1)[:4]),
                               int(m.group(1)[4:6]),
                               int(m.group(1)[6:8]))
                if pc_date < mtg_date:
                    prev_pc = pc
                    prev_pc_date = pc_date
                    break

        # ── Agent IB: Monthly Bulletin più recente pre-blackout ──
        all_mb = sorted(cfg.MONTHLY_BULL_DIR.glob("mb*.pdf"))
        bulletin_path = None
        bulletin_date_str = mtg
        for mb in reversed(all_mb):
            m = re.search(r'mb(\d{6})en\.pdf', mb.name)
            if m:
                try:
                    mb_date = date(int(m.group(1)[:4]),
                                   int(m.group(1)[4:6]), 28)
                    if mb_date < blackout:
                        bulletin_path = mb
                        bulletin_date_str = mb_date.isoformat()
                        break
                except:
                    continue

        if not prev_pc:
            log.warning(f"No press conference found for {mtg} — skipping")
            continue

        # ── Run agents ────────────────────────────────────────
        im_out = run_agent_im(
            meeting_date_str=mtg,
            account_path=None,           # no Accounts pre-2015
            press_conf_path=prev_pc,
            prev_meeting_date_str=prev_pc_date.isoformat() if prev_pc_date else '',
            account_pub_date_str=prev_pc_date.isoformat() if prev_pc_date else '',
        )
        if not im_out:
            log.warning(f"Agent IM returned None for {mtg} — skipping")
            continue

        ib_out = run_agent_ib(
            target_meeting_str=mtg,
            bulletin_path=bulletin_path,
            bulletin_pub_date_str=bulletin_date_str,
            doc_type="ECB Monthly Bulletin",
        )
        if not ib_out:
            log.warning(f"Agent IB returned None for {mtg} — skipping")
            continue

        ii_out = run_agent_ii(im_out, ib_out, mtg)
        if not ii_out:
            continue

        if mtg not in policy_df.index:
            log.warning(f"No policy decision for {mtg}")
            continue

        row = policy_df.loc[mtg]
        rolling_std = None
        if len(results) >= 6:
            past_mech = [r['surprises']['dfr_surprise']['surprise_mech']
                         for r in results[-12:] if 'surprises' in r]
            if past_mech:
                rolling_std = float(np.std(past_mech))

        surprises = compute_surprises(
            ii_output=ii_out,
            actual_dfr_bp=float(row.get('dfr_change_bp', 0)),
            actual_mro_bp=float(row.get('mro_change_bp', 0)),
            actual_mlf_bp=float(row.get('mlf_change_bp', 0)),
            rolling_std=rolling_std,
        )

        record = {
            "meeting_date": mtg,
            "agent_im": im_out,
            "agent_ib": ib_out,
            "agent_ii": ii_out,
            "surprises": surprises,
        }
        results.append(record)

        cp_path = cfg.CHECKPOINTS_DIR / f"{mtg}.json"
        with open(cp_path, 'w') as f:
            json.dump(record, f, indent=2)

        log.info(f"  ✅ {mtg} — DFR surprise: "
                 f"{surprises['dfr_surprise']['surprise_mech']:.1f}bp "
                 f"({surprises['dfr_surprise']['direction']})")

    return results

print("✅ run_pipeline_pre2015 defined")

In [ ]:
PRE2015_MEETINGS = sorted([d for d in policy_df.index if d < '2015-01-01'])
print(f"Meeting da processare: {len(PRE2015_MEETINGS)}")
results_pre2015 = run_pipeline_pre2015(PRE2015_MEETINGS)

In [ ]:
# Override extract_html_text — usa Introductory Statement per Press Conference
def extract_html_text(path: Path) -> str:
    """
    Per le Press Conference: estrae solo l'Introductory Statement.
    Per altri HTML: estrae tutto il testo.
    """
    if 'pressconf_' in path.name:
        return extract_introductory_statement(path)
    try:
        with open(path, 'r', encoding='utf-8', errors='ignore') as f:
            soup = BeautifulSoup(f.read(), 'html.parser')
        for tag in soup(['nav', 'footer', 'header', 'script', 'style']):
            tag.decompose()
        text = soup.get_text(separator='\n')
        text = re.sub(r'\n{3,}', '\n\n', text)
        return text.strip()
    except Exception as e:
        log.warning(f"HTML extraction failed for {path.name}: {e}")
        return ""

print("✅ extract_html_text aggiornata")

In [ ]:
# ============================================================
# FIX: Parser Introductory Statement only
# ============================================================

def extract_introductory_statement(path: Path) -> str:
    """
    Estrae solo l'Introductory Statement dalla Press Conference HTML,
    escludendo il Q&A che inizia dopo 'We are now at your disposal for questions.'
    """
    try:
        with open(path, 'r', encoding='utf-8', errors='ignore') as f:
            soup = BeautifulSoup(f.read(), 'html.parser')
        for tag in soup(['nav', 'footer', 'header', 'script', 'style']):
            tag.decompose()
        text = soup.get_text(separator='\n')
        text = re.sub(r'\n{3,}', '\n\n', text).strip()

        separators = [
            'We are now at your disposal for questions.',
            'We are now at your disposal for your questions.',
            'I am now at your disposal for questions.',
            'I am now at your disposal for your questions.',
            'QUESTIONS AND ANSWERS',
            'Question:',
            'QUESTION:',
        ]

        for sep in separators:
            idx = text.find(sep)
            if idx > 500:
                text = text[:idx].strip()
                break

        text = re.sub(r'\n{3,}', '\n\n', text)
        return text.strip()

    except Exception as e:
        log.warning(f"HTML extraction failed for {path.name}: {e}")
        return ""

print("✅ extract_introductory_statement definita")

In [ ]:
# Rilancia batch pre-2015 con Introductory Statement
PRE2015_MEETINGS = sorted([d for d in policy_df.index if d < '2015-01-01'])
print(f"Meeting da processare: {len(PRE2015_MEETINGS)}")
print(f"Primo: {PRE2015_MEETINGS[0]} — Ultimo: {PRE2015_MEETINGS[-1]}")

results_pre2015 = run_pipeline_pre2015(PRE2015_MEETINGS)